# Phases 3 and 4 — fine-tune Qwen3-8B, then benchmark it

Section 5 trains (Phase 3). Section 6 benchmarks against the sealed test set (Phase 4).

Thin driver. All logic lives in the repo (`scripts/train_student.py`, `src/pii/eval.py`) so it is
version-controlled and diffable; a notebook that holds the logic cannot be reviewed or reverted.

**Session settings:** Accelerator `GPU T4 x2`, Internet `On`, Persistence `Variables and Files`.
**Secrets required:** `HF_TOKEN`, `WANDB_API_KEY` — both must be *attached* to this notebook.
**Dataset required:** `pii-distillation-data`, added via Add Input.

Run cells 1–4 once per session, then one cell from section 5 or 6.

> For the two ~5h runs use **Save Version → Save & Run All**, not the interactive session:
> it runs headless on Kaggle's servers, so a closed laptop is irrelevant.

## 1 · Repo and dependencies

In [ ]:
# Step out before deleting: on a re-run the kernel's cwd IS /kaggle/working/repo, and removing the
# directory you are standing in leaves every later command failing on getcwd — including pip, which
# reports the confusing "folder you are executing pip from can no longer be found".
%cd /kaggle/working

# Idempotent on purpose: `git clone` into an existing non-empty directory fails rather than updating,
# and under `-q` that error is easy to miss — leaving a stale checkout that looks fine until a fix you
# pushed appears not to have worked.
!rm -rf /kaggle/working/repo
!git clone -q --branch phase-4 https://github.com/eren-o23/model-distillation-pipeline.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -1

# torch is NOT in requirements-train.txt: Kaggle's image ships a CUDA build matched to its driver,
# and replacing it costs 2.5GB and may not load.
!pip install -q -r requirements-train.txt

# Print the resolved versions rather than just importing. Kaggle preinstalls several of these, so a pin
# that is looser than the real floor leaves the old build in place and fails much later, at model load,
# as an ImportError that names a version nobody chose.
!python -c "import torch,transformers,peft,bitsandbytes,accelerate; print(f'torch {torch.__version__} cuda {torch.version.cuda} | transformers {transformers.__version__} | peft {peft.__version__} | bitsandbytes {bitsandbytes.__version__} | accelerate {accelerate.__version__}')"

## 2 · Data

`data/*` is gitignored, so the clone arrives empty. Symlinking the attached dataset into the repo's
`data/` keeps `DATA_DIR` and `load_split()` working unchanged — no paths threaded through the code.

In [ ]:
from pathlib import Path

# The mount path is discovered, not hardcoded. Kaggle slugifies the dataset title, so the folder name
# need not match what you typed, and any subfolder structure from the upload is preserved.
ROOT_IN = Path('/kaggle/input')
DST = Path('/kaggle/working/repo/data')
DST.mkdir(exist_ok=True)

found = {p.name: p for p in ROOT_IN.rglob('*.jsonl')}
if not found:
    listing = '\n  '.join(str(p) for p in sorted(ROOT_IN.rglob('*'))[:40]) or '(nothing mounted)'
    raise SystemExit(
        f'No .jsonl files under /kaggle/input. What is actually mounted:\n  {listing}\n'
        'Add the dataset via Add Input, then Run > Restart session.'
    )

# test.jsonl joins the list in Phase 4. It was deliberately absent through Phases 2 and 3, and it
# has to be added to the Kaggle dataset as a new version before this cell can find it.
for name in ('train_sft.jsonl', 'val_sft.jsonl', 'val.jsonl', 'test.jsonl'):
    target = found.get(name)
    assert target, f'{name} not found. Mounted .jsonl files: {sorted(found)}'
    link = DST / name
    link.unlink(missing_ok=True)
    link.symlink_to(target)
    print(f'{name:20} {target.stat().st_size / 1e6:7.1f} MB   <- {target.parent}')

# The seal is open, so it is replaced by proof rather than removed: this must be the file Phase 1
# froze, not one rebuilt since. The benchmark scripts re-check it before every run; failing here
# instead costs seconds rather than an hour of GPU.
import sys
sys.path.insert(0, '/kaggle/working/repo')
from src.pii.data import verify_frozen

for split in ('val', 'test'):
    print(f'{split}.jsonl sha256 {verify_frozen(split)[:16]}… matches the manifest')


## 3 · Secrets

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

s = UserSecretsClient()
os.environ['HF_TOKEN'] = s.get_secret('HF_TOKEN')
os.environ['WANDB_API_KEY'] = s.get_secret('WANDB_API_KEY')

# The 16GB base model must not land in /kaggle/working, which is capped at 20GB and is also where
# checkpoints go. /kaggle/temp is scratch with far more room.
os.environ['HF_HOME'] = '/kaggle/temp/hf'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Silence the download and weight-loading progress bars. They write thousands of carriage-returned
# lines into the committed log, which pushed the actual error past the log viewer's limit and made a
# one-line argparse failure take a round trip to find.
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'

print('secrets loaded:', all(os.environ.get(k) for k in ('HF_TOKEN', 'WANDB_API_KEY')))

## 4 · Preflight

Everything here fails in seconds if it is going to fail at all. The point is to never discover a
sm_75 or prompt-rendering problem five hours into a run.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!python -m pytest tests/ -q

## 5 · Runs — Phase 3, complete

**Every run cell below is commented out.** Phase 3 is finished and its adapters are on the
Hub; left live, a Save & Run All aimed at section 6 would spend 14h retraining them. Uncomment
a single line if a configuration genuinely needs re-running.

Calibrated on this hardware: **24.44 s/step**, per-device batch 4 x accum 4, training peak 12.13 GiB and
eval peak 9.87 GiB of 14.56. Two epochs is 7.0h and fits a session; three is 10.5h and does not (D-024).

Split across **two sessions** so neither needs `--resume`:

| session | cells | approx |
|---|---|---|
| A | baselines + rank 8 | ~7.5h |
| B | rank 32 + final full-val eval | ~7.9h |

Use **Save Version → Save & Run All** for these, not the interactive session — it runs headless on
Kaggle's servers, so a closed laptop is irrelevant. Watch progress at
[wandb.ai/ereno23-university-of-exeter/pii-distillation](https://wandb.ai/ereno23-university-of-exeter/pii-distillation).

In [ ]:
# Optional smoke test (~15 min). Only needed after a change to the training code.
# !python -u scripts/train_student.py --rank 8 --limit 256 --epochs 1 --no-push

# --- SESSION A ---------------------------------------------------------------------------------
# Baselines: untuned Qwen3-8B. Answers "how much did fine-tuning buy over just prompting?"
# ~12 min each.
# !python -u scripts/train_student.py --baseline short
# !python -u scripts/train_student.py --baseline teacher

In [ ]:
# Config A — rank 8. ~7.0h. Pushes the best epoch to erenrosman/pii-qwen3-8b-lora-r8.
# !python -u scripts/train_student.py --rank 8

In [ ]:
# --- SESSION B ---------------------------------------------------------------------------------
# Config B — rank 16, NOT rank 32. Rank 32's 87M-param adapter OOMs at batch 4 on a 14.56GiB T4,
# and running it would have required batch 2, which changes more than the rank (D-026). Rank 16
# keeps batch 4 x accum 4 identical to rank 8, so rank really is the only variable. ~7h.
#
# Comments stay on their own line: a trailing "#" glued to an argument gave
# "argument --eval-n: invalid int value: '0#'" and cost a whole session.
# !python -u scripts/train_student.py --rank 16

In [ ]:
# Final: the winning adapter over all 1,000 val rows, for the headline number (~50 min).
# Set the repo to whichever rank won on the 200-row per-epoch evals.
# !python -u scripts/train_student.py --eval-adapter erenrosman/pii-qwen3-8b-lora-r8 --eval-n 0

In [ ]:
# Final: the winning adapter over all 1,000 val rows, for the headline number.
# !python -u scripts/train_student.py --eval-adapter erenrosman/pii-qwen3-8b-lora-r8 --eval-n 0

## 6 · Collect

`reports/raw/phase3/*.json` is what `scripts/write_phase3.py` turns into the report, so these files
must come back off Kaggle. They are small.

In [ ]:
!mkdir -p /kaggle/working/out && cp -r /kaggle/working/repo/reports/raw/phase3 /kaggle/working/out/
!ls -la /kaggle/working/out/phase3/

## 6 · Phase 4 — benchmark against the sealed test set

The test split is opened here and nowhere else. Both quality runs score all 1,000 rows through
`src.pii.eval.evaluate()` unchanged, so these numbers and Phase 3's val numbers come from one harness.

| cell | run | approx |
|---|---|---|
| A | rank 8 quality, 1,000 test rows | ~50 min |
| B | rank 16 quality, 1,000 test rows | ~50 min |
| C | latency and throughput, batch 1/8/16/32 | ~30 min |

All three fit one session. The teacher half was already measured from the laptop — 0.822 micro-F1 for
$0.46 — and is committed, so nothing here spends money.

In [ ]:
# A — the deployed adapter on all 1,000 test rows. The headline quality number.
!python -u scripts/benchmark_student.py --adapter erenrosman/pii-qwen3-8b-lora-r8

In [ ]:
# B — rank 16 on the same rows. D-027 called the rank comparison null on 200 val rows and flagged
# that it might be underpowered rather than genuinely null; 1,000 unseen rows is what settles it.
!python -u scripts/benchmark_student.py --adapter erenrosman/pii-qwen3-8b-lora-r16

In [ ]:
# C — latency and throughput on the same seeded 100 rows the teacher was timed on.
# Batch 32 is expected to OOM: eval peaked at 9.87 of 14.56 GiB at batch 16. That is the card's
# ceiling and gets recorded as a row in the curve rather than avoided.
!python -u scripts/benchmark_student.py --adapter erenrosman/pii-qwen3-8b-lora-r8 --latency

### Collect

`reports/raw/phase4/*.json` is what `scripts/write_phase4.py` turns into the report. Download it
**before the session ends** — `/kaggle/working` does not reliably survive, and Phase 3 lost a full
eval that way.

In [ ]:
!mkdir -p /kaggle/working/out && cp -r /kaggle/working/repo/reports/raw/phase4 /kaggle/working/out/
!ls -la /kaggle/working/out/phase4/